# RSM Django Site Scraper
## Sites: mgta495 and mgta455

These sites use a standard username + password login form (no SSO required).

### How web authentication works
1. You GET the login page → server sends back a **CSRF token** (a security value to prevent fake form submissions)
2. You POST your credentials + CSRF token → server validates and sets a **session cookie** (`sessionid`)
3. Every subsequent request automatically includes that cookie → server knows who you are

`requests.Session()` handles cookies automatically, so steps 2→3 are seamless.

## Step 1: Install dependencies
Run this once if you haven't already.

In [ ]:
# !pip install requests beautifulsoup4 python-dotenv

## Step 2: Set your credentials

The login field is named `login` (not `username`) — this is standard for Django sites using [django-allauth](https://django-allauth.readthedocs.io/), which accepts email as the identifier.

> **Security tip:** Don't hardcode your password in the notebook. Use `getpass` so it's never stored in cell output or `.ipynb` history.

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env into os.environ

BASE_URL = "https://rsm-django-02.ucsd.edu"

EMAIL = os.getenv("RSM_EMAIL")
PASSWORD = os.getenv("RSM_PASSWORD")

if not EMAIL or not PASSWORD:
    raise RuntimeError("RSM_EMAIL or RSM_PASSWORD not set in .env file")

## Step 3: Authenticate

Here's exactly what happens:
1. `session.get(login_url)` — fetches the login page; Django sets a `csrftoken` cookie automatically
2. We extract the CSRF token value from that cookie
3. `session.post(login_url, data={...})` — sends credentials + CSRF token; Django validates and sets `sessionid`
4. All future `session.get()` calls automatically include `sessionid` — we're logged in

In [ ]:
def login(course_slug, email, password):
    """Authenticate to rsm-django-02.ucsd.edu for the given course prefix."""
    session = requests.Session()
    login_url = f"{BASE_URL}/{course_slug}/accounts/login/"

    # Step 1: GET the login page — this gives us the csrftoken cookie
    session.get(login_url)

    # Step 2: Extract CSRF token from cookies
    csrf_token = session.cookies.get("csrftoken")
    if not csrf_token:
        raise RuntimeError("Could not retrieve CSRF token from login page")

    # Step 3: POST credentials
    payload = {
        "csrfmiddlewaretoken": csrf_token,  # required by Django to prevent CSRF attacks
        "login": email,                      # field name is 'login', not 'username'
        "password": password,
        "remember": "on",                   # keeps session alive longer
    }
    headers = {"Referer": login_url}  # Django requires this for CSRF validation

    response = session.post(login_url, data=payload, headers=headers)

    # Step 4: Verify we're logged in (should NOT be on the login page anymore)
    if "accounts/login" in response.url:
        raise RuntimeError("Login failed — check your email/password")

    print(f"[{course_slug}] Logged in as {email}")
    return session


# Create one session per course (they are separate Django apps)
session_495 = login("mgta495", EMAIL, PASSWORD)
session_455 = login("mgta455", EMAIL, PASSWORD)

## Step 4: Explore the site structure

Before scraping, use these cells to understand what's on each page.
Open your browser's DevTools (F12 → Elements) to identify the right HTML tags.

In [ ]:
def list_links(session, url, filter_keyword=None):
    """Print all links on a page, optionally filtered by keyword in href."""
    r = session.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        if filter_keyword is None or filter_keyword in href:
            print(f"{text:50s}  {href}")

# List all links on the mgta495 home page
print("=== MGTA 495 home page links ===")
list_links(session_495, f"{BASE_URL}/mgta495/")

In [ ]:
# Inspect a single module page to see what content is there
r = session_495.get(f"{BASE_URL}/mgta495/module01/")
soup = BeautifulSoup(r.text, "html.parser")

print("=== Links on Module 01 ===")
for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(strip=True)
    # Focus on non-nav links
    if text and len(text) > 3:
        print(f"  {text:50s}  {href}")

In [ ]:
# Check the student files page
r = session_495.get(f"{BASE_URL}/mgta495/student/files/")
soup = BeautifulSoup(r.text, "html.parser")
print(r.status_code)
print(soup.prettify()[:3000])

## Step 5: Download files

The pages have module pages (`/mgta495/module01/` through `/mgta495/module10/`) and a student files page (`/mgta495/student/files/`).

Run the exploration cells above first to see the actual link patterns, then adjust `file_link_filter` below.

In [ ]:
ROOT_DIR = "./Courses"


def make_safe_filename(name):
    return "".join(c for c in name if c not in r'\/:*?"<>|').strip()


def download_file(session, url, save_path):
    r = session.get(url, stream=True)
    r.raise_for_status()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"    Saved: {save_path}")


def scrape_page_for_files(session, page_url, save_dir):
    """
    Fetch a page and download any links that look like files.
    Adjust the filter below based on what you see in the explore cells.
    """
    r = session.get(page_url)
    if r.status_code != 200:
        print(f"  Skipped {page_url} (status {r.status_code})")
        return

    soup = BeautifulSoup(r.text, "html.parser")

    FILE_EXTENSIONS = (".pdf", ".xlsx", ".xls", ".csv", ".zip", ".ipynb", ".py", ".docx", ".pptx")

    downloaded = 0
    for a in soup.find_all("a", href=True):
        href = a["href"]

        # Only download links that look like actual files
        is_file = (
            any(href.lower().endswith(ext) for ext in FILE_EXTENSIONS)
            or "/download" in href
            or "/files/" in href
        )
        if not is_file:
            continue

        # Build absolute URL
        if href.startswith("http"):
            file_url = href
        elif href.startswith("/"):
            file_url = BASE_URL + href
        else:
            file_url = page_url.rstrip("/") + "/" + href

        # Derive a filename
        link_text = a.get_text(strip=True)
        url_filename = href.split("/")[-1].split("?")[0]
        filename = make_safe_filename(link_text or url_filename) or url_filename
        # Ensure file has an extension
        if "." not in filename and "." in url_filename:
            filename += "." + url_filename.rsplit(".", 1)[-1]

        save_path = os.path.join(save_dir, filename)
        try:
            download_file(session, file_url, save_path)
            downloaded += 1
        except Exception as e:
            print(f"    FAILED: {file_url} — {e}")

    if downloaded == 0:
        print(f"  No downloadable files found on {page_url}")

In [ ]:
import os

ROOT_DIR = "./Courses"
FILE_EXTS = (".pdf", ".xlsx", ".xls", ".csv", ".zip", ".ipynb", ".py", ".docx", ".pptx", ".txt", ".json")


def download_file(session, url, save_path):
    r = session.get(url, stream=True)
    r.raise_for_status()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    size_kb = os.path.getsize(save_path) / 1024
    print(f"    ✓ {os.path.basename(save_path)} ({size_kb:.0f} KB)")


def save_page_text(session, url, save_path):
    """Save the readable text content of a page (objectives, tasks, rubrics, etc.)."""
    from bs4 import BeautifulSoup
    r = session.get(url)
    if r.status_code != 200:
        return
    soup = BeautifulSoup(r.text, "html.parser")
    main = soup.find("main") or soup.body
    lines = []
    for tag in (main or soup).find_all(["h1","h2","h3","h4","p","li"]):
        t = tag.get_text(strip=True)
        if t and len(t) > 3:
            prefix = {"h1": "# ", "h2": "## ", "h3": "### ", "h4": "#### "}.get(tag.name, "- " if tag.name == "li" else "")
            lines.append(prefix + t)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write("\n".join(lines))
    print(f"    ✓ {os.path.basename(save_path)} (page text)")


# ─── MGTA 495: GenAI for Business ───────────────────────────────────────────
# Files: /mgta495/student/files/download/module0X/<filename>
# Text:  /mgta495/module0X/, /mgta495/assignment0X/

course_dir_495 = os.path.join(ROOT_DIR, "MGTA495-GenAI-for-Business")

# Module files + page text
modules_495 = {
    "module01": ["module01-GenAI-forms.pdf", "module01-tt.pdf", "module01.pdf"],
    "module02": ["customer-ticket-process.pdf", "module02.pdf"],
    "module03": ["module03.pdf", "webscraping.zip"],
    "module05": ["module05.pdf"],
    "module06": ["module06-ai-workflows.pdf", "module06-from-skill-to-tools-to-mcp.pdf",
                 "module06-how-llms-work.pdf", "module06.pdf"],
    "module07": ["module07-RAG.pdf", "module07-how-llms-work.pdf", "module07-workflow.pdf"],
    "module08": ["module07-RAG.pdf", "module08.pdf"],
    "module09": ["ai-higher-ed-scaffolding-critical-thinking.pdf", "module09.pdf"],
}

print("=== MGTA 495 — Downloading Files ===")
for mod, filenames in modules_495.items():
    mod_dir = os.path.join(course_dir_495, mod)
    print(f"\n[{mod}]")
    for fname in filenames:
        url = f"{BASE_URL}/mgta495/student/files/download/{mod}/{fname}"
        save_path = os.path.join(mod_dir, fname)
        try:
            download_file(session_495, url, save_path)
        except Exception as e:
            print(f"    ✗ {fname}: {e}")

print("\n=== MGTA 495 — Saving Page Text (objectives, tasks, rubrics) ===")
# Modules
for i in range(1, 11):
    mod = f"module{i:02d}"
    url = f"{BASE_URL}/mgta495/{mod}/"
    save_path = os.path.join(course_dir_495, mod, f"{mod}-page.txt")
    save_page_text(session_495, url, save_path)

# Assignments (rubrics + deliverables)
for i in range(1, 5):
    assign = f"assignment{i:02d}"
    url = f"{BASE_URL}/mgta495/{assign}/"
    save_path = os.path.join(course_dir_495, "assignments", f"{assign}.txt")
    save_page_text(session_495, url, save_path)

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ─── MGTA 455: AI-Assisted Customer Analytics ────────────────────────────────
# All files centralized at /mgta455/downloads/file/<filename>
# Discoverable from /mgta455/downloads/

course_dir_455 = os.path.join(ROOT_DIR, "MGTA455-Customer-Analytics")

print("=== MGTA 455 — Discovering all files from /downloads/ ===")
r = session_455.get(f"{BASE_URL}/mgta455/downloads/")
soup = BeautifulSoup(r.text, "html.parser")

download_files = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/downloads/file/" in href:
        filename = href.split("/")[-1]
        full_url = urljoin(BASE_URL, href)
        download_files.append((filename, full_url))

print(f"Found {len(download_files)} files")
for fname, url in download_files:
    print(f"  {fname}")

print("\n=== MGTA 455 — Downloading all files ===")
files_dir = os.path.join(course_dir_455, "files")
for fname, url in download_files:
    save_path = os.path.join(files_dir, fname)
    try:
        download_file(session_455, url, save_path)
    except Exception as e:
        print(f"    ✗ {fname}: {e}")

print("\n=== MGTA 455 — Saving Page Text (objectives, case descriptions, rubrics) ===")
# Module pages
for i in range(1, 12):
    mod = f"module{i:02d}"
    url = f"{BASE_URL}/mgta455/{mod}/"
    save_path = os.path.join(course_dir_455, mod, f"{mod}-page.txt")
    save_page_text(session_455, url, save_path)

# Case pages (real business cases with datasets)
for i in range(1, 9):
    case = f"case{i:02d}"
    url = f"{BASE_URL}/mgta455/{case}/"
    save_path = os.path.join(course_dir_455, "cases", f"{case}.txt")
    save_page_text(session_455, url, save_path)

# Resource pages (syllabus, grading policy, GenAI policy)
for page in ["syllabus", "grading-policy", "genai-policy", "computing"]:
    url = f"{BASE_URL}/mgta455/resources/{page}/"
    save_path = os.path.join(course_dir_455, "resources", f"{page}.txt")
    save_page_text(session_455, url, save_path)

print("\nDone!")

---
## What you learned: the general pattern for any Django site

```
session = requests.Session()
session.get(login_url)                          # 1. get CSRF token
csrf = session.cookies["csrftoken"]
session.post(login_url, data={                  # 2. submit credentials
    "csrfmiddlewaretoken": csrf,
    "login": email,
    "password": password,
}, headers={"Referer": login_url})
session.get(protected_url)                      # 3. all subsequent requests are authenticated
```

The field names (`login` vs `username`) and login URL vary by site — always inspect the `<form>` in DevTools first.

### Security reminders
- Use `getpass.getpass()` instead of hardcoding passwords — this prevents them from being saved in notebook output
- The Canvas ACCESS_TOKEN in `canvas_download_script.ipynb` is hardcoded — if you ever push that notebook to GitHub, rotate that token immediately at Canvas → Account → Settings → Access Tokens